# Python Exceptions

Exceptions report errors that occur while a program runs. This notebook covers the core syntax once, then applies it in complete, runnable examples.

## 1. What an exception is

A syntax error prevents Python from parsing code. An exception is an object raised while valid code runs. If nothing handles it, Python stops the current call stack and prints a traceback. Common built-in types include `ValueError`, `TypeError`, `KeyError`, `IndexError`, `ZeroDivisionError`, `FileNotFoundError`, and other `OSError` subclasses.

In [ ]:
examples = [ValueError, TypeError, KeyError, IndexError, ZeroDivisionError, FileNotFoundError]
for exception_type in examples:
    print(f"{exception_type.__name__:22} -> {exception_type.__base__.__name__}")

## 2. `try` and `except`

Put only the operation that may fail in `try`. Catch the most specific exception you can actually recover from. Execution continues after the complete statement.

In [ ]:
text = "not-a-number"
try:
    number = int(text)
except ValueError as error:
    print(f"Cannot convert {text!r}: {error}")

print("Program continues")

## 3. Multiple handlers and grouped exceptions

Python checks handlers from top to bottom, so place specific subclasses before broader parent classes. A tuple groups exception types only when they need the same recovery.

In [ ]:
def divide_from_text(left, right):
    try:
        result = float(left) / float(right)
    except ValueError as error:
        return f"Invalid number: {error}"
    except ZeroDivisionError:
        return "The divisor must not be zero"
    except TypeError as error:
        return f"Expected strings or numbers: {error}"
    return result

for values in [("12", "3"), ("x", "3"), ("12", "0"), (None, 2)]:
    print(values, "->", divide_from_text(*values))

In [ ]:
def lookup(sequence, position):
    try:
        return sequence[position]
    except (IndexError, TypeError) as error:
        return f"Lookup failed ({type(error).__name__}): {error}"

print(lookup([10, 20], 8))
print(lookup([10, 20], "one"))

## 4. The four blocks: `try`, `except`, `else`, `finally`

- `try`: runs the risky operation.
- `except`: runs only for a matching exception.
- `else`: runs only when `try` finishes without an exception. Keep success-only work here.
- `finally`: runs whether the operation succeeds, fails, returns, or uses `break`/`continue`; use it for required cleanup.

At least one `except` or `finally` block is required. `else` requires `except`.

In [ ]:
def parse_percentage(text):
    try:
        value = float(text)
    except (TypeError, ValueError) as error:
        print("except: conversion failed", error)
        return None
    else:
        print("else: conversion succeeded")
        return value / 100
    finally:
        print("finally: attempt completed")

print("Result:", parse_percentage("25"))
print("Result:", parse_percentage("bad"))

## 5. `finally` for cleanup

The cleanup block runs even when the function returns or an exception propagates. Context managers such as `with open(...)` are usually the cleaner choice for files, locks, and connections.

In [ ]:
class DemoResource:
    def open(self):
        print("resource opened")
        return self

    def read(self):
        print("resource used")
        return "data"

    def close(self):
        print("resource closed")


def use_resource():
    resource = DemoResource().open()
    try:
        return resource.read()
    finally:
        resource.close()

print("Result:", use_resource())

Never `return`, `break`, or `continue` from `finally`: it can hide an active exception or replace an earlier return value. Cleanup should normally be the only job of this block.

## 6. Raising exceptions

Use `raise ExceptionType(message)` when a function cannot fulfill its contract. Choose a standard exception when it describes the problem; `ValueError` is appropriate when a value has the correct type but an invalid value.

In [ ]:
def set_discount(percent):
    if isinstance(percent, bool) or not isinstance(percent, (int, float)):
        raise TypeError("percent must be a number")
    if not 0 <= percent <= 100:
        raise ValueError("percent must be between 0 and 100")
    return percent / 100

for candidate in (20, 125, "20"):
    try:
        print(candidate, "->", set_discount(candidate))
    except (TypeError, ValueError) as error:
        print(candidate, "->", type(error).__name__, error)

## 7. Re-raising and exception chaining

A bare `raise` inside a handler re-raises the same exception with its traceback. `raise NewError(...) from error` translates an implementation-level error while preserving the cause. Use `from None` only when deliberately hiding irrelevant implementation detail.

In [ ]:
def parse_age(record):
    try:
        return int(record["age"])
    except KeyError as error:
        raise ValueError("record is missing 'age'") from error
    except (TypeError, ValueError) as error:
        raise ValueError("age must contain a whole number") from error

for record in ({"age": "42"}, {}, {"age": "unknown"}):
    try:
        print(parse_age(record))
    except ValueError as error:
        cause = type(error.__cause__).__name__ if error.__cause__ else None
        print(f"{error}; caused by {cause}")

In [ ]:
def log_and_reraise():
    try:
        int("invalid")
    except ValueError:
        print("Conversion failed; recording context before propagation")
        raise

try:
    log_and_reraise()
except ValueError:
    print("Caller handled the re-raised exception")

## 8. Custom exceptions

Create a custom type when callers need to distinguish a domain error from ordinary built-ins. Inherit from `Exception`, give related errors a common base, and use an `Error` suffix.

In [ ]:
class OrderError(Exception):
    """Base class for order-processing errors."""


class OutOfStockError(OrderError):
    def __init__(self, product, requested, available):
        self.product = product
        self.requested = requested
        self.available = available
        super().__init__(
            f"{product}: requested {requested}, only {available} available"
        )


def reserve(product, quantity, inventory):
    if quantity <= 0:
        raise ValueError("quantity must be positive")
    available = inventory.get(product, 0)
    if quantity > available:
        raise OutOfStockError(product, quantity, available)
    inventory[product] -= quantity
    return inventory[product]

stock = {"keyboard": 3}
try:
    remaining = reserve("keyboard", 5, stock)
except OutOfStockError as error:
    print(error)
    print("Available units:", error.available)
else:
    print("Remaining units:", remaining)

## 9. Complete example: validating a batch

Handle an error at the level that can make a useful decision. Here, one invalid row is rejected while the rest of the batch continues.

In [ ]:
class RecordError(Exception):
    pass


def normalize_record(record):
    try:
        name = record["name"].strip()
        score = float(record["score"])
    except KeyError as error:
        raise RecordError(f"missing field: {error.args[0]}") from error
    except (AttributeError, TypeError, ValueError) as error:
        raise RecordError("name and score have invalid values") from error

    if not name:
        raise RecordError("name cannot be empty")
    if not 0 <= score <= 100:
        raise RecordError("score must be between 0 and 100")
    return {"name": name.title(), "score": score}


records = [
    {"name": " anu ", "score": "91.5"},
    {"name": "Bala"},
    {"name": "Chen", "score": 105},
    {"name": "Divya", "score": 88},
]

accepted, rejected = [], []
for row_number, record in enumerate(records, start=1):
    try:
        accepted.append(normalize_record(record))
    except RecordError as error:
        rejected.append({"row": row_number, "reason": str(error)})

print("Accepted:", accepted)
print("Rejected:", rejected)

## 10. Assertions are not input validation

`assert condition` raises `AssertionError` and is useful for internal invariants and debugging. Python can disable assertions with optimization, so use ordinary checks and explicit exceptions for user input, API data, and business rules.

In [ ]:
def average(values):
    assert values, "internal invariant: values must not be empty"
    return sum(values) / len(values)

print(average([10, 20, 30]))

## 11. Good practices and common mistakes

- Catch specific exceptions; avoid bare `except:` because it also catches shutdown signals such as `KeyboardInterrupt`.
- Do not use `except Exception: pass`; silent failure makes defects difficult to diagnose.
- Keep `try` blocks small so handlers do not catch unrelated failures.
- Do not catch an error unless you can recover, add useful context, or clean up and re-raise.
- Preserve tracebacks with bare `raise`; translate layers with `raise ... from error`.
- Prefer `with` for managed resources and reserve `finally` for cleanup that has no context manager.
- Exception handling is for exceptional paths, not normal branching.
- Never place control-flow statements in `finally` when they could suppress an exception.

## Quick reference

```python
try:
    value = risky_operation()
except SpecificError as error:
    recover(error)
except (AnotherError, RelatedError) as error:
    recover_another_way(error)
else:
    use(value)                 # only when try succeeded
finally:
    cleanup()                  # always runs
```

In [ ]:
assert divide_from_text("12", "3") == 4.0
assert parse_age({"age": "42"}) == 42
assert accepted == [
    {"name": "Anu", "score": 91.5},
    {"name": "Divya", "score": 88.0},
]
assert len(rejected) == 2
print("All exception examples completed successfully.")